## Name : Tejas Jiddewar
## Batch : C4_54
## Sem : VI
## NLP Lab : 8

In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
import seaborn as sns
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix

In [ ]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

In [ ]:
df['sentiment'].value_counts()

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess(text):

    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]

    return " ".join(words)

In [ ]:
df["clean_review"] = df["review"].apply(preprocess)

df.head()

In [ ]:
bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df["clean_review"])

print(X_bow.shape)

In [ ]:
bow_df = pd.DataFrame(X_bow.toarray(), columns=bow.get_feature_names_out())

bow_df.head()

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df["clean_review"])
y = df["sentiment"]
y = y.map({"positive":1, "negative":0})
feature_names = tfidf.get_feature_names_out()

print(feature_names)

In [ ]:
tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df.head()

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

In [ ]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

In [ ]:
svm = LinearSVC()
svm.fit(X_train, y_train)

svm_pred = svm.predict(X_test)

In [ ]:
def evaluate_model(name, y_test, y_pred):

    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    return [name, acc, pre, rec, f1]

In [ ]:
results = []

results.append(evaluate_model("Logistic Regression", y_test, lr_pred))
results.append(evaluate_model("Naive Bayes", y_test, nb_pred))
results.append(evaluate_model("SVM", y_test, svm_pred))

results_df = pd.DataFrame(results, columns=[
    "Model","Accuracy","Precision","Recall","F1 Score"
])

results_df

In [ ]:
cm = confusion_matrix(y_test,svm_pred)

sns.heatmap(cm,annot=True,fmt="d")

plt.title("Confusion Matrix")
plt.show()

In [ ]:
import pickle

pickle.dump(svm,open("sentiment_model.pkl","wb"))

pickle.dump(vectorizer,open("tfidf_vectorizer.pkl","wb"))

print("Model saved")

In [ ]:
from google.colab import files

files.download("sentiment_model.pkl")

files.download("tfidf_vectorizer.pkl")

In [ ]:
!pip install streamlit pyngrok

In [ ]:
%%writefile app.py

import streamlit as st
import pickle
import numpy as np

# Page configuration
st.set_page_config(
    page_title="Movie Sentiment Analyzer",
    page_icon="🎬",
    layout="centered"
)

# Custom CSS for styling
st.markdown("""
<style>
.main {
background-color:#0f172a;
}
.title {
color:#38bdf8;
font-size:40px;
text-align:center;
font-weight:bold;
}
.subtitle{
text-align:center;
color:#cbd5f5;
font-size:18px;
}
.result-box{
padding:20px;
border-radius:10px;
text-align:center;
font-size:24px;
font-weight:bold;
}
</style>
""", unsafe_allow_html=True)

# Load model
model = pickle.load(open("sentiment_model.pkl","rb"))
vectorizer = pickle.load(open("tfidf_vectorizer.pkl","rb"))

# Header
st.markdown(
    '<p class="title" style="font-size:40px; font-weight:bold;">🎬 Movie Review Sentiment Analyzer</p>',
    unsafe_allow_html=True
)

st.markdown(
    '<p class="subtitle" style="font-size:20px;">AI powered NLP system to detect positive or negative movie reviews</p>',
    unsafe_allow_html=True
)

st.divider()

# Input area
review = st.text_area("✍️ Enter Movie Review", height=150)

col1, col2 = st.columns(2)

predict = col1.button(" Predict Sentiment")

if predict:

    if review.strip() == "":
        st.warning("⚠️ Please enter a movie review")

    else:

        review_vec = vectorizer.transform([review])

        prediction = model.predict(review_vec)
        decision = model.decision_function(review_vec)

        confidence = round(abs(decision[0]) * 10, 2)

        st.divider()

        # Sentiment result
        if prediction[0] == 1:

            st.markdown(
                f'<div class="result-box" style="background-color:#16a34a;color:white;">😊 Positive Review</div>',
                unsafe_allow_html=True
            )

        else:

            st.markdown(
                f'<div class="result-box" style="background-color:#dc2626;color:white;">😡 Negative Review</div>',
                unsafe_allow_html=True
            )

        st.write("")

        # Confidence
        st.metric("Prediction Confidence", f"{confidence}%")

        st.progress(min(confidence/100,1.0))

        st.write("")

        # Review stats
        st.subheader(" Review Analysis")

        words = review.split()

        col1, col2, col3 = st.columns(3)

        col1.metric("Total Words", len(words))
        col2.metric("Characters", len(review))
        col3.metric("Average Word Length", round(np.mean([len(w) for w in words]),2))

st.divider()



In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("33pMGlSr8fbzwOIcSjNM18oJqNa_32oFJBiEYMCw3bDMnLPFF")

In [ ]:
from pyngrok import ngrok

# stop old tunnels
ngrok.kill()

# start new tunnel
public_url = ngrok.connect(8501)

print("Streamlit URL:", public_url)